In [ ]:
!unzip ./Data_to_Students-20260702T040614Z-3-001.zip

Archive:  ./Data_to_Students-20260702T040614Z-3-001.zip
  inflating: Data_to_Students/test_en_1000.csv  
  inflating: Data_to_Students/dev_sa_1000.csv  
  inflating: Data_to_Students/test_sa_1000.csv  
  inflating: Data_to_Students/train_sa_10000.csv  
  inflating: Data_to_Students/dev_en_1000.csv  
  inflating: Data_to_Students/train_en_10000.csv  


In [1]:
#@title Install Dependencies
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install transformers bert-score sacremoses nltk tf-keras accelerate>=1.1.0

Looking in indexes: https://download.pytorch.org/whl/cu118



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
#@title Data Loading and Model Setup
import pandas as pd
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from bert_score import score
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from tqdm.auto import tqdm

data_path = 'Data_to_Students/'
test_df = pd.merge(pd.read_csv(data_path + 'test_sa_1000.csv'), pd.read_csv(data_path + 'test_en_1000.csv'), on='Source_id')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_name = 'facebook/nllb-200-distilled-600M'
print("Using device:",device)
tokenizer = AutoTokenizer.from_pretrained(model_name, src_lang="san_Deva")
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

C:\Users\sufya\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


Loading weights: 100%|██████████| 512/512 [00:00<00:00, 44625.82it/s]


In [2]:
#@title Translation Inference
import time
from tqdm.auto import tqdm

def translate_batch(texts, model, tokenizer, batch_size=8):
    model.eval()
    results = []
    start_time = time.time()

    # Use tqdm to track progress over batches
    for i in tqdm(range(0, len(texts), batch_size), desc="Translating Batches"):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors='pt', padding=True, truncation=True).to(device)
        with torch.no_grad():
            # Correct way to get the language ID for NLLB
            target_lang_id = tokenizer.convert_tokens_to_ids("eng_Latn")
            generated_tokens = model.generate(
                **inputs,
                forced_bos_token_id=target_lang_id,
                max_length=128
            )
        results.extend(tokenizer.batch_decode(generated_tokens, skip_special_tokens=True))

    end_time = time.time()
    inference_time = end_time - start_time
    return results, inference_time

print(f"Starting inference on {len(test_df)} examples...")
predictions, inference_time_taken = translate_batch(test_df['Sentence_sa'].tolist(), model, tokenizer)

print(f"\nInference completed in {inference_time_taken:.2f} seconds.")
print("Sample Predictions:")
for i in range(3):
    print(f"Source: {test_df['Sentence_sa'].iloc[i]}")
    print(f"Result: {predictions[i]}\n")

Starting inference on 1000 examples...


Translating Batches: 100%|██████████| 125/125 [01:19<00:00,  1.57it/s]


Inference completed in 79.42 seconds.
Sample Predictions:
Source: एक्लिप्स् इति प्रोग्रामर् कृते दोषान्वेषणे अपि साहाय्यं करोति।
Result: Eclipse also helps in troubleshooting the program.

Source: विश्वासकारणादेव समभाषि मया वचः। इति यथा शास्त्रे लिखितं तथैवास्माभिरपि विश्वासजनकम् आत्मानं प्राप्य विश्वासः क्रियते तस्माच्च वचांसि भाष्यन्ते।
Result: We also have the same spirit of faith and will speak, just as it is written: "I spoke with faith".

Source: तदा, तत्स्वयं ड्रैवर निमित्तम् अन्वेष्यति। अहं 'Cancel' इत्यस्योपरि नुदामि।
Result: Then he will look for his own driver. I will not cancel.



In [3]:
import time

smoothie = SmoothingFunction().method1
refs = [[s.split()] for s in test_df['Sentence_en']]
hyps = [s.split() for s in predictions]
bleu_val = corpus_bleu(refs, hyps, smoothing_function=smoothie) * 100

P, R, F1 = score(predictions, test_df['Sentence_en'].tolist(), lang='en', device=device, rescale_with_baseline=True)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'BLEU Score: {bleu_val:.2f}')
print(f'BERTScore F1: {F1.mean().item():.4f}')
print(f'Inference Time: {inference_time_taken:.2f} seconds')
print(f'Total Model Parameters: {total_params:,}')

submission = pd.DataFrame({
    'Source_id': test_df['Source_id'],
    'Sentence_en': predictions
})
submission.to_csv('submission.csv', index=False)
display(submission.head())

Loading weights: 100%|██████████| 389/389 [00:00<00:00, 4604.79it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BLEU Score: 10.97
BERTScore F1: 0.4675
Inference Time: 79.42 seconds
Total Model Parameters: 615,073,792


,Source_id,Sentence_en
0,1,Eclipse also helps in troubleshooting the prog...
1,2,We also have the same spirit of faith and will...
2,3,Then he will look for his own driver. I will n...
3,4,"For all iterations, the iterator is set to the..."
4,5,"When he opened the second seal, I heard a voic..."


In [4]:

analysis_df = pd.DataFrame({
    'Source (Sanskrit)': test_df['Sentence_sa'].head(10),
    'Reference (English)': test_df['Sentence_en'].head(10),
    'Prediction': predictions[:10]
})

print("--- Translation Examples for Qualitative Analysis ---")
display(analysis_df)

--- Translation Examples for Qualitative Analysis ---


,Source (Sanskrit),Reference (English),Prediction
0,एक्लिप्स् इति प्रोग्रामर् कृते दोषान्वेषणे अपि...,Eclipse also helps the programmer to find out ...,Eclipse also helps in troubleshooting the prog...
1,विश्वासकारणादेव समभाषि मया वचः। इति यथा शास्त्...,"""We having the same spirit of faith, according...",We also have the same spirit of faith and will...
2,"तदा, तत्स्वयं ड्रैवर निमित्तम् अन्वेष्यति। अहं...",Then it will automatically begin searching for...,Then he will look for his own driver. I will n...
3,"सर्वेभ्यः इटरेशन्-अर्थम्, iterator इतीदं प्रत्...",The iterator will be set to each of the indice...,"For all iterations, the iterator is set to the..."
4,अपरं द्वितीयमुद्रायां तेन मोचितायां द्वितीयस्य...,"""And when he had opened the second seal, I hea...","When he opened the second seal, I heard a voic..."
5,"वयम्, ओब्जेक्ट्स् स्वकीयान् स्थितीन् फील्ड्स्...",We know that objects store their individual st...,We found out that objects store their position...
6,बाल: युष्मासु विश्वासं करोति ।,Boy has belief on you all.,Child: He believes in you.
7,प्रकृति का अवलोकन करने से आप आश्चर्य चकित हो स...,Observing the nature helps create a sense of w...,You may be amazed at the beauty of nature.
8,यूयं कीदृक् तस्याज्ञा अपालयत भयकम्पाभ्यां तं ग...,"""And his inward affection is more abundant tow...",And his affection for you is growing more and ...
9,"""वर्षाया: जलस्य कश्चन अंश: भूमौ शुष्को भूत्वा ...",A part of rain water is absorbedby earth. Wate...,"""In the rainy season, if a part of the water f..."


## Final Evaluation and Submission
This section computes the official metrics required by the assignment guidelines (Section 6) and exports the results to `submission.csv`.

In [5]:
import time
from bert_score import score
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

# 1. Efficiency Metrics
total_params = sum(p.numel() for p in model.parameters())
print(f"--- Efficiency Metrics ---")
print(f"Total Model Parameters: {total_params:,}")
print(f"Total Inference Time (Test Set): {inference_time_taken:.2f} seconds\n")

# 2. BLEU Score (Default NLTK BLEU with no weights as per Section 6)
smoothie = SmoothingFunction().method1
# NLTK default BLEU uses geometric mean of 1-4 grams (weights=[0.25, 0.25, 0.25, 0.25])
bleu_final = corpus_bleu(refs, hyps, smoothing_function=smoothie) * 100

# 3. BERTScore (F1 with rescale_with_baseline=True)
P, R, F1 = score(predictions, test_df['Sentence_en'].tolist(), lang='en', device=device, rescale_with_baseline=True)
bert_f1_final = F1.mean().item()

print(f"--- Accuracy Metrics ---")
print(f"BLEU Score: {bleu_final:.2f}")
print(f"BERTScore F1: {bert_f1_final:.4f}")

--- Efficiency Metrics ---
Total Model Parameters: 615,073,792
Total Inference Time (Test Set): 79.42 seconds



Loading weights: 100%|██████████| 389/389 [00:00<00:00, 11429.02it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


--- Accuracy Metrics ---
BLEU Score: 10.97
BERTScore F1: 0.4675


In [6]:
# 4. Generate Final Submission CSV
submission_final = pd.DataFrame({
    'Source_id': test_df['Source_id'],
    'Sentence_en': predictions
})

# Ensure UTF-8 encoding as per Section 5
submission_final.to_csv('submission.csv', index=False, encoding='utf-8')

print("Submission file 'submission.csv' has been generated and saved.")
display(submission_final.head())

Submission file 'submission.csv' has been generated and saved.


,Source_id,Sentence_en
0,1,Eclipse also helps in troubleshooting the prog...
1,2,We also have the same spirit of faith and will...
2,3,Then he will look for his own driver. I will n...
3,4,"For all iterations, the iterator is set to the..."
4,5,"When he opened the second seal, I heard a voic..."


In [7]:
import pandas as pd
import os

# Define the full paths for the new files
new_en_file = r'.\test_demo_en - Sheet1.csv'
new_sa_file = r'.\test_demo_sa - Sheet1.csv'

print(f"Attempting to load data from: {new_en_file} and {new_sa_file}")

try:
    # Load the new data from files into DataFrames
    new_test_en_df = pd.read_csv(new_en_file)
    new_test_sa_df = pd.read_csv(new_sa_file)

    # Merge the new English and Sanskrit dataframes on Source_id
    new_user_test_df = pd.merge(new_test_sa_df, new_test_en_df, on='Source_id', suffixes=('_sa', '_en'))

    display(new_user_test_df.head())
    print(f"Loaded {len(new_user_test_df)} new user-provided test examples for Sanskrit to English translation.")
except FileNotFoundError as e:
    print(f"Error: {e}\nMake sure to upload your CSV files to the '{user_files_path}' directory.")
    new_user_test_df = pd.DataFrame() # Create an empty DataFrame to prevent errors in subsequent cells

Attempting to load data from: .\test_demo_en - Sheet1.csv and .\test_demo_sa - Sheet1.csv


,Source_id,Sentence_sa,Sentence_en
0,1001,हे भ्रातरो युष्माकम् आत्माभिमानो यन्न जायते तद...,"""For I would not, brethren, that ye should be ..."
1,1002,लाभा: पादयो: स्नायु: दृढा तथा आकृति: सम्यक भवत...,• Leg muscles become shapely and stronger. • I...
2,1003,अस्मिन् पाठे वयं : उपयोक्त्रे admin role असैन्...,"In this tutorial, we learnt how to: assign adm..."
3,1004,"वयमस्मिन् course पृष्टे यदा भवामः तदा, वामतः ...","While we are on this course page, notice that ..."
4,1005,केल्क्युलेस् कोर्स् इत्यस्मै संयोक्तुं ये इच्छ...,This list has students that I want to enroll t...


Loaded 100 new user-provided test examples for Sanskrit to English translation.


In [9]:
if not new_user_test_df.empty:
    # Perform Sanskrit to English translation on the new test set

    print(f"Starting Sanskrit to English translation on {len(new_user_test_df)} new user-provided examples...")

    # Reuse the existing translate_batch function, model, tokenizer (configured for san_Deva to eng_Latn) and device.
    # The 'model', 'tokenizer', and 'device' are expected to be available from cell cd701f37.
    new_user_predictions, new_user_inference_time = translate_batch(
        new_user_test_df['Sentence_sa'].tolist(), 
        model, 
        tokenizer, 
    )

    print(f"\nSanskrit to English translation for new user-provided data completed in {new_user_inference_time:.2f} seconds.")
    print("Sample New User-Provided Predictions:")
    for i in range(min(3, len(new_user_test_df))):
        print(f"Source (Sanskrit): {new_user_test_df['Sentence_sa'].iloc[i]}")
        print(f"Result (English): {new_user_predictions[i]}\n")
else:
    print("Skipping translation: new_user_test_df is empty due to previous file loading error.")

Starting Sanskrit to English translation on 100 new user-provided examples...


Translating Batches: 100%|██████████| 13/13 [00:07<00:00,  1.71it/s]


Sanskrit to English translation for new user-provided data completed in 7.61 seconds.
Sample New User-Provided Predictions:
Source (Sanskrit): हे भ्रातरो युष्माकम् आत्माभिमानो यन्न जायते तदर्थं ममेदृशी वाञ्छा भवति यूयं एतन्निगूढतत्त्वम् अजानन्तो यन्न तिष्ठथ; वस्तुतो यावत्कालं सम्पूर्णरूपेण भिन्नदेशिनां संग्रहो न भविष्यति तावत्कालम् अंशत्वेन इस्रायेलीयलोकानाम् अन्धता स्थास्यति;
Result (English): For I do not want you to be ignorant of this mystery, brothers, so that you may be spiritually blind and spiritually ignorant: The Israelites will be blinded for a time until the full number of the Gentiles will be gathered together.

Source (Sanskrit): लाभा: पादयो: स्नायु: दृढा तथा आकृति: सम्यक भवति ।एष: जङ्घाया: स्नायुं समस्याभ्य: रक्षति ।
Result (English): Benefits: The legs: muscles: strength and shape: consistency.

Source (Sanskrit): अस्मिन् पाठे वयं : उपयोक्त्रे admin role असैन् कर्तुं, course इत्यस्मै teacher असैन् कर्तुं, course मध्ये student नामाङ्कनकरणं कथमिति च ज्ञातवन्तः ।
Result (

In [12]:
if not new_user_test_df.empty:
    # Re-calculate evaluation metrics for the new user-provided test set
    from bert_score import score
    from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

    smoothie = SmoothingFunction().method1

    # Prepare references and hypotheses for BLEU calculation
    new_user_refs = [[s.split()] for s in new_user_test_df['Sentence_en'].tolist()]
    new_user_hyps = [s.split() for s in new_user_predictions]

    # Calculate BLEU Score
    new_user_bleu_val = corpus_bleu(new_user_refs, new_user_hyps, smoothing_function=smoothie) * 100

    # Calculate BERTScore
    P_new_user, R_new_user, F1_new_user = score(
        new_user_predictions, 
        new_user_test_df['Sentence_en'].tolist(), 
        lang='en', 
        device=device, 
        rescale_with_baseline=True
    )
    new_user_bert_f1_final = F1_new_user.mean().item()

    print(f"--- Evaluation Metrics for New User-Provided Test Set ---")
    # Assuming total_params is available from previously run cells (like 87076f15)
    print(f"Total Model Parameters: {total_params:,}")
    print(f"BLEU Score: {new_user_bleu_val:.2f}")
    print(f"BERTScore F1: {new_user_bert_f1_final:.4f}")
    print(f"Inference Time (New User-Provided Test Set): {new_user_inference_time:.2f} seconds")
else:
    print("Skipping evaluation: new_user_test_df is empty.")

Loading weights: 100%|██████████| 389/389 [00:00<00:00, 8645.90it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


--- Evaluation Metrics for New User-Provided Test Set ---
Total Model Parameters: 615,073,792
BLEU Score: 10.63
BERTScore F1: 0.4441
Inference Time (New User-Provided Test Set): 7.61 seconds


In [13]:
if not new_user_test_df.empty:
    # Generate a new submission CSV for the new user-provided test set predictions
    new_user_submission_df = pd.DataFrame({
        'Source_id': new_user_test_df['Source_id'],
        'Sentence_en': new_user_predictions
    })

    # Ensure UTF-8 encoding
    new_user_submission_df.to_csv('new_user_submission.csv', index=False, encoding='utf-8')

    print("New submission file 'new_user_submission.csv' has been generated and saved with predictions from the new user-provided test set.")
    display(new_user_submission_df.head())
else:
    print("Skipping submission file generation: new_user_test_df is empty.")

New submission file 'new_user_submission.csv' has been generated and saved with predictions from the new user-provided test set.


,Source_id,Sentence_en
0,1001,For I do not want you to be ignorant of this m...
1,1002,Benefits: The legs: muscles: strength and shap...
2,1003,In this lesson we learned: how to make a user ...
3,1004,"When we get to the course, the menu on the lef..."
4,1005,Students who wish to take a course in calculus...
